# Contract Clause Risk Analysis — QLoRA Fine-tuning
**Model:** LLaMA 3.1 8B Instruct  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**Task:** Role-conditioned risk classification (Licensor / Licensee → Low / Medium / High)

### Before running:
1. **Accelerator** → Settings → Accelerator → **GPU T4 x2** (or P100)
2. **HF Token** → Add a Kaggle Secret named `HF_TOKEN` with your Hugging Face token (Notebook → Add-ons → Secrets). Request LLaMA 3.1 access at hf.co/meta-llama if needed.
3. **Dataset files** → Upload `train.jsonl` and `val.jsonl` as a Kaggle Dataset (+ Add data → New Dataset), then attach it. They will appear under `/kaggle/input/<dataset-name>/`.
   - Update `TRAIN_FILE` / `VAL_FILE` in the Config cell to match your dataset path.
4. **Internet** → Must be **ON** (Settings → Internet → On) to pull model weights from HF.
5. Outputs are saved to `/kaggle/working/qlora_output/` — download them from the Output tab when done.

## 1. Install Dependencies
Kaggle kernels ship with PyTorch pre-installed — we only need the QLoRA/LoRA packages.
Run this cell once and wait for it to finish before continuing.

In [ ]:
%%capture
!pip install -q \
    transformers==4.44.2 \
    peft==0.12.0 \
    trl==0.8.6 \
    bitsandbytes==0.43.1 \
    accelerate==0.33.0 \
    datasets==2.21.0 \
    huggingface-hub==0.23.4 \
    scikit-learn \
    seaborn \
    matplotlib
print('✓ Packages installed')

## 2. Hugging Face Login
Reads your HF token from **Kaggle Secrets** (key: `HF_TOKEN`). Never hard-code tokens in notebooks.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('✓ Logged in to Hugging Face')

## 3. Config — all hyperparameters in one place

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────
# Update these to match your Kaggle Dataset path
TRAIN_FILE  = "/kaggle/input/your-dataset-name/train.jsonl"  # ← edit dataset name
VAL_FILE    = "/kaggle/input/your-dataset-name/val.jsonl"    # ← edit dataset name
OUTPUT_DIR  = "/kaggle/working/qlora_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ──────────────────────────────────────────────────────
MODEL_ID       = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# ── QLoRA ──────────────────────────────────────────────────────
LORA_RANK      = 16    # size of the adapter matrices; higher = more capacity
LORA_ALPHA     = 32    # scaling factor; rule of thumb: 2x rank
LORA_DROPOUT   = 0.05
LORA_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Training ───────────────────────────────────────────────────
EPOCHS         = 3
BATCH_SIZE     = 2     # per device; keep low for T4
GRAD_ACCUM     = 8     # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LR             = 2e-4
MAX_SEQ_LEN    = 512
WARMUP_RATIO   = 0.05

SEED           = 42

print(f'✓ Config set. Outputs → {OUTPUT_DIR}')
print(f'  Train: {TRAIN_FILE}')
print(f'  Val:   {VAL_FILE}')

## 4. Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE}
)

print(dataset)
print("\nSample training example:")
print(dataset["train"][0]["text"])

## 5. Load Model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NormalFloat4 — best for normally distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16 for stability
    bnb_4bit_use_double_quant=True,         # quantize quantization constants too (~0.4 GB saved)
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token  # LLaMA has no pad token by default
tokenizer.padding_side = "right"           # pad on right for causal LM training

print(f"Model loaded. Parameters: {model.num_parameters()/1e9:.1f}B")

## 6. Attach LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized model for training
# (enables gradient checkpointing, casts layer norms to float32)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5–1% of total params — that's the LoRA adapter size

## 7. Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",   # set to "wandb" if you want experiment tracking
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
)

print("Starting training...")
trainer.train()
print("Training complete.")

## 8. Save LoRA Adapter Weights
Weights are saved to `/kaggle/working/qlora_output/`. Download them from the **Output** tab on the right after the notebook finishes.

In [ ]:
import os, json as _json, datetime

# ── Save adapter + tokenizer ───────────────────────────────────
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ── Write a small run-info file so you remember what this checkpoint is ──
run_info = {
    "model_id": MODEL_ID,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "max_seq_len": MAX_SEQ_LEN,
    "saved_at": datetime.datetime.now().isoformat(),
}
with open(os.path.join(OUTPUT_DIR, "run_info.json"), "w") as f:
    _json.dump(run_info, f, indent=2)

# ── Confirm files saved ───────────────────────────────────────
saved_files = os.listdir(OUTPUT_DIR)
print(f"\n✓ Adapter saved to: {OUTPUT_DIR}")
print(f"  Files: {sorted(saved_files)}")
print("\nTo reload later:")
print(f"  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{OUTPUT_DIR}')")

## 9. Evaluate — Extract Predictions on Val Set

We run inference on each val example, parse the predicted risk label out of the generated text, and compare against the ground truth.

In [ ]:
import json
import re
from tqdm import tqdm

def extract_risk_label(generated_text: str) -> str:
    """
    Parse 'Risk Level: High' from model output.
    Returns 'Unknown' if parsing fails.
    """
    match = re.search(r"Risk Level:\s*(Low|Medium|High)", generated_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    # fallback: look for standalone label
    match = re.search(r"\b(Low|Medium|High)\b", generated_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "Unknown"


def get_prompt_only(full_text: str) -> str:
    """
    Strip the assistant response from the full training text,
    leaving just the prompt up to the assistant header.
    """
    marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    idx = full_text.find(marker)
    if idx != -1:
        return full_text[:idx + len(marker)]
    return full_text


def get_ground_truth_label(full_text: str) -> str:
    """Extract the ground truth label from the assistant response."""
    marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    idx = full_text.find(marker)
    if idx != -1:
        response = full_text[idx + len(marker):]
        return extract_risk_label(response)
    return "Unknown"


# Load val examples
val_examples = []
with open(VAL_FILE) as f:
    for line in f:
        val_examples.append(json.loads(line.strip()))

print(f"Running inference on {len(val_examples)} validation examples...")

model.eval()

y_true, y_pred, parties = [], [], []

for ex in tqdm(val_examples):
    full_text = ex["text"]
    prompt    = get_prompt_only(full_text)
    gt_label  = get_ground_truth_label(full_text)
    party     = ex.get("_meta", {}).get("party", "Unknown")

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,      # greedy decoding for deterministic eval
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    generated  = tokenizer.decode(new_tokens, skip_special_tokens=True)
    pred_label = extract_risk_label(generated)

    y_true.append(gt_label)
    y_pred.append(pred_label)
    parties.append(party)

print("Inference complete.")

## 10. Performance Metrics

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

LABELS = ["Low", "Medium", "High"]

# ── Overall metrics ───────────────────────────────────────────
print("=" * 55)
print("OVERALL PERFORMANCE (val set)")
print("=" * 55)
print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
print()
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

# ── Per-party metrics ─────────────────────────────────────────
for party in ["Licensor", "Licensee"]:
    idx = [i for i, p in enumerate(parties) if p == party]
    if not idx:
        continue
    pt = [y_true[i] for i in idx]
    pp = [y_pred[i] for i in idx]
    print(f"{'=' * 55}")
    print(f"{party.upper()} (n={len(idx)})")
    print(f"{'=' * 55}")
    print(f"Accuracy: {accuracy_score(pt, pp):.3f}")
    print(classification_report(pt, pp, labels=LABELS, zero_division=0))

In [ ]:
# ── Confusion matrices ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

splits = [
    ("Overall",  list(range(len(y_true))), y_true, y_pred),
    ("Licensor", [i for i,p in enumerate(parties) if p=="Licensor"], None, None),
    ("Licensee", [i for i,p in enumerate(parties) if p=="Licensee"], None, None),
]

for ax, (title, idx, yt, yp) in zip(axes, splits):
    if yt is None:
        yt = [y_true[i] for i in idx]
        yp = [y_pred[i] for i in idx]
    cm = confusion_matrix(yt, yp, labels=LABELS)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=LABELS, yticklabels=LABELS, ax=ax,
        vmin=0, vmax=1,
    )
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j+0.5, i+0.72, f"({cm[i,j]})",
                    ha="center", va="center", fontsize=8, color="gray")
    ax.set_title(f"{title} (n={len(yt)})", fontsize=13)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle("Confusion Matrices — Fine-tuned LLaMA 3.1 8B (QLoRA)", fontsize=14, y=1.02)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, "confusion_matrices.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")

In [ ]:
# ── High↔Low flip analysis ────────────────────────────────────
flips = [
    (i, y_true[i], y_pred[i], parties[i])
    for i in range(len(y_true))
    if set([y_true[i], y_pred[i]]) == {"High", "Low"}
]

print(f"High↔Low flips: {len(flips)}")
if flips:
    print(f"{'Idx':>4}  {'True':>7}  {'Pred':>7}  Party")
    print("-" * 35)
    for idx, true, pred, party in flips:
        print(f"{idx:>4}  {true:>7}  {pred:>7}  {party}")

In [ ]:
# ── Prediction distribution ───────────────────────────────────
from collections import Counter

print("Prediction distribution (val):")
pred_dist = Counter(y_pred)
true_dist = Counter(y_true)
print(f"{'Label':>8}  {'True':>6}  {'Pred':>6}")
print("-" * 26)
for label in LABELS:
    print(f"{label:>8}  {true_dist[label]:>6}  {pred_dist[label]:>6}")

medium_frac = pred_dist.get("Medium", 0) / len(y_pred)
print(f"\nMedium prediction rate: {medium_frac:.1%}")
if medium_frac > 0.6:
    print("⚠️  Model may still be collapsing toward Medium — consider more epochs or checking annotations.")
else:
    print("✓  No medium-collapse detected.")

## 11. Qualitative Check — Sample Predictions

In [ ]:
import textwrap

def show_sample(idx):
    ex = val_examples[idx]
    prompt = get_prompt_only(ex["text"])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=120, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True)

    party  = ex.get("_meta", {}).get("party", "?")
    true_l = y_true[idx]
    pred_l = y_pred[idx]
    match  = "✓" if true_l == pred_l else "✗"

    print(f"{'─'*60}")
    print(f"Example #{idx}  |  Party: {party}  |  True: {true_l}  |  Pred: {pred_l}  {match}")
    print(f"{'─'*60}")
    print("Generated output:")
    print(textwrap.fill(generated.strip(), width=80))
    print()

correct_idx   = next((i for i in range(len(y_true)) if y_true[i] == y_pred[i]), 0)
incorrect_idx = next((i for i in range(len(y_true)) if y_true[i] != y_pred[i]), 1)
high_idx      = next((i for i in range(len(y_true)) if y_true[i] == "High"), 2)

for idx in [correct_idx, incorrect_idx, high_idx]:
    show_sample(idx)